In [1]:
import ast
import json
from openai import OpenAI
from pydantic._internal._model_construction import ModelMetaclass
from typing import Dict, Any, Union

from src.pydantic_models.step2_transcription import TranscribedOldJapanese
from src.utils.openai_utils import openai_api_call
from src.utils.io import save_results

In [2]:
SYSTEM_PROMPT_1_2 = """
# Old Japanese Transcription

## Step 1: Character Annotation

You are working on Old Japanese Man'yōgana usage.

**Task:**
For each graph in the input text, determine whether it functions as:

- **PHON** = phonograph
- **LOG** = logograph
- **UNC** = uncertain

Use surrounding context, corpus conventions, and known Man'yōgana patterns.

**Important:**
- Analyze each graph in context.
- If evidence is insufficient, choose UNC.

---

## Step 2: Phonemic Transcription

Convert the input Man'yōgana text into Old Japanese phonemic transcription.

**Syllabic Structure:** (Consonant) + Vowel

**When multiple readings are possible, follow this order:**
1. Established lexical or philological reading in tradition
2. Corpus-attested usage
3. Annotation (PHON / LOG / UNC)

**Rules:**
- Preserve morpheme boundaries whenever recoverable.
- Preserve token order.
- Do NOT invent unattested forms unless clearly marked uncertain.
- If graph sequence forms a known lexical item, prioritize whole-word reading over character-by-character reading.

---

## Example

**Input:**
```
朝布麻須等六
```

**Output:**
```json
{
  "text": [
    {
      "char": "朝",
      "type": "LOG",
      "transcription": [
        {
          "vowel": "a",
          "consonant": null
        },
        {
          "vowel": "a",
          "consonant": "s"
        }
      ]
    },
    {
      "char": "布",
      "type": "PHON",
      "transcription": [
        {
          "vowel": "u",
          "consonant": "p"
        }
      ]
    },
    {
      "char": "麻",
      "type": "PHON",
      "transcription": [
        {
          "vowel": "a",
          "consonant": "m"
        }
      ]
    },
    {
      "char": "須",
      "type": "PHON",
      "transcription": [
        {
          "vowel": "u",
          "consonant": "s"
        }
      ]
    },
    {
      "char": "等",
      "type": "PHON",
      "transcription": [
        {
          "vowel": "o",
          "consonant": "t"
        }
      ]
    },
    {
      "char": "六",
      "type": "PHON",
      "transcription": [
        {
          "vowel": "u",
          "consonant": "m"
        }
      ]
    }
  ],
  "reasoning": null
}
```
"""

In [4]:
def pretty_print_json(obj: Union[str, dict]) -> None:
    """
    Pretty print a JSON object with proper formatting.

    Args:
        obj: Dictionary or string representation of a dictionary
    """
    if isinstance(obj, dict) or isinstance(obj, list):
        json_obj = obj
    else:
        json_obj = ast.literal_eval(obj)
    pretty_json = json.dumps(json_obj, ensure_ascii=False, indent=4).replace('\\n', '\n')
    print(pretty_json)

def test_api_call(
    client: OpenAI,
    model: str,
    input_txt: str,
    system_prompt: str,
    output_schema: ModelMetaclass,
    max_retries: int = 3,
    base_delay: float = 0.05,
    max_delay: float = 60.0,
    **api_args
) -> None:
    """
    Test an OpenAI API call on a single row from a DataFrame.

    This function is useful for debugging and testing API calls before
    running the full batch processing. It displays both input and output
    in a formatted way.

    Args:
        client: OpenAI client instance
        model: Model name to use
        system_prompt: System prompt for the model
        raw_text_input: Display raw text instead of pretty JSON for input (default: False)
        max_retries: Maximum number of retry attempts for API calls (default: 3)
        base_delay: Initial delay between retries in seconds (default: 0.05)
        max_delay: Maximum delay between retries in seconds (default: 60.0)
        **api_args: Additional arguments passed to OpenAI API

    Example:
        >>> test_api_call(
        ...     client=OpenAI(),
        ...     df=df,
        ...     model="gpt-4o",
        ...     system_prompt="Extract policy information",
        ...     schema=PolicyMetadata,
        ...     row=42
        ... )
    """
    api_params = {
        "model": model,
        "input": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": input_txt}
        ],
        "text_format": output_schema,
        **api_args
    }

    print("\nInput:\n")

    print(input_txt)

    response = openai_api_call(
        client, api_params,
        max_retries=max_retries,
        base_delay=base_delay,
        max_delay=max_delay
    )

    print("\n" + "=" * 80)
    print("\nOutput:\n")
    outputs = response.output_parsed.model_dump()
    pretty_print_json(outputs)


In [6]:
client = OpenAI()

MODEL_NAME = "gpt-5.2"
extra_args = {
    "reasoning": {"effort": "high"}
}

In [7]:
test_api_call(
    client=client,
    input_txt="寧樂乃家爾者",
    model=MODEL_NAME,
    system_prompt=SYSTEM_PROMPT_1_2,
    output_schema=TranscribedOldJapanese
)


Input:

寧樂乃家爾者


Output:

{
    "text": [
        {
            "char": "寧",
            "transcription": [
                {
                    "vowel": "i",
                    "consonant": null
                }
            ],
            "type": "LOG"
        },
        {
            "char": "樂",
            "transcription": [
                {
                    "vowel": "a",
                    "consonant": "r"
                },
                {
                    "vowel": "a",
                    "consonant": null
                }
            ],
            "type": "LOG"
        },
        {
            "char": "乃",
            "transcription": [
                {
                    "vowel": "o",
                    "consonant": "n"
                }
            ],
            "type": "PHON"
        },
        {
            "char": "家",
            "transcription": [
                {
                    "vowel": "e",
                    "consonant": "y"
                